In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pyserial scikit-learn scipy pandas numpy joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from scipy.fftpack import fft
from sklearn.ensemble import IsolationForest
import joblib
import os

In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/training_data.zip"
extract_path = "/content/training_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully")

✅ Extracted successfully


In [ ]:
DATA_FOLDER = "/content/training_data/training_data"

all_features = []

for filename in sorted(os.listdir(DATA_FOLDER)):
    if filename.endswith(".csv"):
        df = pd.read_csv(os.path.join(DATA_FOLDER, filename))
        all_features.append(df)

print(f"✅ Loaded {len(all_features)} samples")

✅ Loaded 50 samples


In [ ]:
def extract_features(df):
    features = []

    for col in ["ax", "ay", "az", "gx", "gy", "gz"]:
        signal = df[col].values
        N = len(signal)

        # FFT
        fft_vals = np.abs(fft(signal))[:N//2]
        freqs = np.fft.fftfreq(N, d=0.01)[:N//2]  # 100Hz sampling

        # 8-12 Hz band energy (tremor band)
        band_mask = (freqs >= 8) & (freqs <= 12)
        band_energy = np.sum(fft_vals[band_mask] ** 2)

        # Peak frequency
        peak_freq = freqs[np.argmax(fft_vals)]

        features.extend([band_energy, peak_freq])

    return features

feature_matrix = []
for df in all_features:
    feature_matrix.append(extract_features(df))

X = np.array(feature_matrix)
print(f"✅ Feature matrix shape: {X.shape}")

✅ Feature matrix shape: (50, 12)


In [ ]:
model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

model.fit(X)
print("✅ Model trained successfully!")

✅ Model trained successfully!


In [ ]:
predictions = model.predict(X)
normal = np.sum(predictions == 1)
anomaly = np.sum(predictions == -1)

print(f"Normal samples: {normal}")
print(f"Anomaly samples: {anomaly}")
print(f"Accuracy on training data: {(normal/len(predictions))*100:.1f}%")

Normal samples: 47
Anomaly samples: 3
Accuracy on training data: 94.0%


In [ ]:
joblib.dump(model, "/content/drive/MyDrive/morpholock_model.pkl")
print("✅ Model saved → morpholock_model.pkl")

✅ Model saved → morpholock_model.pkl
